# 03. Data Cleaning with PySpark

---

## 📚 Course Overview

Welcome to **Data Cleaning with PySpark**! This notebook teaches you how to clean and prepare messy real-world data at scale.

### What You'll Learn:
1. [**Why Data Cleaning Matters** – The cost of dirty data](#1)
2. [**Schemas and Validation** – Type safety and structure enforcement](#2)
3. [**Handling Missing Data** – Strategies for nulls and gaps](#3)
4. [**Duplicate Detection** – Finding and removing duplicates](#4)
5. [**Data Type Conversions** – Safe type casting and validation](#5)
6. [**String Cleaning** – Text manipulation and regex](#6)
7. [**Outlier Detection** – Statistical methods](#7)
8. [**Advanced Techniques** – UDFs, complex transformations](#8)
9. [**Performance Optimization** – Caching and efficient operations](#9)

### Prerequisites:
- Notebooks 01 and 02 completed
- Understanding of DataFrames
- Basic SQL knowledge

---

<a id="1"></a>
## 1️⃣ Why Data Cleaning Matters

### The Cost of Dirty Data

Real-world data is **messy**:
- 📊 **80% of data science time** is spent on cleaning data
- 💸 **$15M average annual cost** of poor data quality per organization
- ⚠️ **Wrong decisions** from incorrect data
- 🐛 **Production failures** from unhandled edge cases

### Common Data Quality Issues:

| **Issue** | **Example** | **Impact** |
|-----------|-------------|------------|
| Missing Values | `NULL`, empty strings | Failed joins, wrong averages |
| Duplicates | Same record 3x | Inflated counts |
| Type Mismatches | "123" as string | Sorting/calc errors |
| Outliers | Age = 999 | Skewed statistics |
| Inconsistent Formats | "NY" vs "New York" | Failed grouping |
| Invalid Data | Negative prices | Business rule violations |

### PySpark for Data Cleaning:

```
Why PySpark?
┌────────────────────────────────────┐
│ ✅ Scales to billions of rows     │
│ ✅ Fault-tolerant (auto-recovery) │
│ ✅ Schema validation built-in     │
│ ✅ Distributed across cluster     │
│ ✅ Lazy evaluation optimization   │
└────────────────────────────────────┘
```

In [2]:
# Initialize SparkSession for data cleaning
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .appName("DataCleaningPySpark") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "5") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")
print(f"Python Version: {spark.sparkContext.pythonVer}")

Spark Version: 4.0.4
Python Version: 3.13


<a id="2"></a>
## 2️⃣ Schemas: The Foundation of Data Quality

### Why Schemas Matter

**Without Schema** (Slow + Risky):
- 🐌 Reads data **twice** (once to infer schema)
- ❓ Guesses data types (often wrong)
- ⚠️ Allows invalid data through

**With Schema** (Fast + Safe):
- ⚡ Single data read
- ✅ Validates types on load
- 🚫 Rejects bad data early

### Creating Schemas

In [4]:
# Example 1: Schema Definition
from pyspark.sql.types import *

# Define schema manually for a voter dataset
voter_schema = StructType([
    StructField("id", IntegerType(), False),        # NOT NULL
    StructField("name", StringType(), False),
    StructField("age", IntegerType(), True),        # Nullable
    StructField("city", StringType(), True),
    StructField("vote_date", DateType(), True)
])

# Create sample data
data = [
    (1, "Alice Johnson", 34, "Dallas", None),
    (2, "Bob Smith", 45, "Houston", None),
    (3, "Charlie Brown", 28, "Austin", None)
]

# Create DataFrame with schema
df_voters = spark.createDataFrame(data, schema=voter_schema)
df_voters.printSchema()
df_voters.show()

root
 |-- id: integer (nullable = false)
 |-- name: string (nullable = false)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- vote_date: date (nullable = true)

+---+-------------+---+-------+---------+
| id|         name|age|   city|vote_date|
+---+-------------+---+-------+---------+
|  1|Alice Johnson| 34| Dallas|     NULL|
|  2|    Bob Smith| 45|Houston|     NULL|
|  3|Charlie Brown| 28| Austin|     NULL|
+---+-------------+---+-------+---------+



In [5]:
# Example 2: Schema from existing DataFrame
# Get schema from one DataFrame and apply to another

schema_from_df = df_voters.schema
print("\nExtracted Schema:")
print(schema_from_df)

# Use this schema for new data
more_data = [
    (4, "Diana Prince", 38, "San Antonio", None),
    (5, "Eve Torres", 29, "Fort Worth", None)
]

df_more_voters = spark.createDataFrame(more_data, schema=schema_from_df)
df_more_voters.show()


Extracted Schema:
StructType([StructField('id', IntegerType(), False), StructField('name', StringType(), False), StructField('age', IntegerType(), True), StructField('city', StringType(), True), StructField('vote_date', DateType(), True)])
+---+------------+---+-----------+---------+
| id|        name|age|       city|vote_date|
+---+------------+---+-----------+---------+
|  4|Diana Prince| 38|San Antonio|     NULL|
|  5|  Eve Torres| 29| Fort Worth|     NULL|
+---+------------+---+-----------+---------+



### 📝 Exercise 1: Define and Validate Schema

**Task**: Create a schema for a product catalog.

**Requirements**:
1. Define schema: product_id (int), name (string), price (double), in_stock (boolean)
2. Create sample data
3. Load and display with proper types

In [6]:
# Your code here
from pyspark.sql.types import *

# Step 1: Define schema
product_schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("price", DoubleType(), False),
    StructField("in_stock", BooleanType(), True)
])

# Step 2 & 3: Create and show DataFrame
product_data = [
    (1, "Laptop", 999.99, True),
    (2, "Mouse", 29.99, False),
    (3, "Keyboard", 79.99, True),
]

df_products = spark.createDataFrame(product_data, product_schema)

df_products.printSchema()
df_products.show()

root
 |-- product_id: integer (nullable = false)
 |-- name: string (nullable = false)
 |-- price: double (nullable = false)
 |-- in_stock: boolean (nullable = true)

+----------+--------+------+--------+
|product_id|    name| price|in_stock|
+----------+--------+------+--------+
|         1|  Laptop|999.99|    true|
|         2|   Mouse| 29.99|   false|
|         3|Keyboard| 79.99|    true|
+----------+--------+------+--------+



In [ ]:
# ✅ Solution
from pyspark.sql.types import *

product_schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("price", DoubleType(), False),
    StructField("in_stock", BooleanType(), True)
])

product_data = [
    (1, "Laptop", 999.99, True),
    (2, "Mouse", 29.99, False),
    (3, "Keyboard", 79.99, True),
]

df_products = spark.createDataFrame(product_data, schema=product_schema)

df_products.printSchema()
df_products.show()

<a id="3"></a>
## 3️⃣ Handling Missing Data

### Types of Missing Data:

1. **MCAR** (Missing Completely At Random) - Safe to drop
2. **MAR** (Missing At Random) - Can impute based on other columns
3. **MNAR** (Missing Not At Random) - Need domain knowledge

### Strategies:

| **Strategy** | **When to Use** | **Method** |
|--------------|-----------------|------------|
| **Drop rows** | <5% missing, MCAR | `dropna()` |
| **Fill with constant** | Categorical data | `fillna("Unknown")` |
| **Fill with mean/median** | Numerical data, MAR | `fillna(df.agg(...)` |
| **Forward/Backward fill** | Time series | Window functions |
| **Impute** | Complex patterns | ML imputation |

In [7]:
# Create dataset with missing values
messy_data = [
    ("Alice", 34, "Engineer", 75000),
    ("Bob", None, "Manager", 85000),
    ("Charlie", 28, None, 65000),
    ("Diana", 45, "Director", None),
    ("Eve", None, None, None),
    ("Frank", 32, "Engineer", 70000),
    (None, 29, "Analyst", 60000)
]

df_messy = spark.createDataFrame(messy_data, ["name", "age", "role", "salary"])
print("Original DataFrame with missing data:")
df_messy.show()

# Check for nulls in each column
print("\nNull counts per column:")
df_messy.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_messy.columns
]).show()

Original DataFrame with missing data:
+-------+----+--------+------+
|   name| age|    role|salary|
+-------+----+--------+------+
|  Alice|  34|Engineer| 75000|
|    Bob|NULL| Manager| 85000|
|Charlie|  28|    NULL| 65000|
|  Diana|  45|Director|  NULL|
|    Eve|NULL|    NULL|  NULL|
|  Frank|  32|Engineer| 70000|
|   NULL|  29| Analyst| 60000|
+-------+----+--------+------+


Null counts per column:
+----+---+----+------+
|name|age|role|salary|
+----+---+----+------+
|   1|  2|   2|     2|
+----+---+----+------+



In [8]:
# Strategy 1: Drop rows with ANY nulls
df_no_nulls = df_messy.dropna()
print(f"After dropna(): {df_no_nulls.count()} rows (from {df_messy.count()})")
df_no_nulls.show()

# Strategy 2: Drop rows with nulls in SPECIFIC columns
df_partial = df_messy.dropna(subset=["name", "salary"])
print(f"\nAfter dropna(subset=['name', 'salary']): {df_partial.count()} rows")
df_partial.show()

# Strategy 3: Drop only if ALL columns are null
df_all_null = df_messy.dropna(how="all")
print(f"\nAfter dropna(how='all'): {df_all_null.count()} rows")
df_all_null.show()

After dropna(): 2 rows (from 7)
+-----+---+--------+------+
| name|age|    role|salary|
+-----+---+--------+------+
|Alice| 34|Engineer| 75000|
|Frank| 32|Engineer| 70000|
+-----+---+--------+------+


After dropna(subset=['name', 'salary']): 4 rows
+-------+----+--------+------+
|   name| age|    role|salary|
+-------+----+--------+------+
|  Alice|  34|Engineer| 75000|
|    Bob|NULL| Manager| 85000|
|Charlie|  28|    NULL| 65000|
|  Frank|  32|Engineer| 70000|
+-------+----+--------+------+


After dropna(how='all'): 7 rows
+-------+----+--------+------+
|   name| age|    role|salary|
+-------+----+--------+------+
|  Alice|  34|Engineer| 75000|
|    Bob|NULL| Manager| 85000|
|Charlie|  28|    NULL| 65000|
|  Diana|  45|Director|  NULL|
|    Eve|NULL|    NULL|  NULL|
|  Frank|  32|Engineer| 70000|
|   NULL|  29| Analyst| 60000|
+-------+----+--------+------+



In [9]:
# Strategy 4: Fill with constants
df_filled = df_messy.fillna({
    "name": "Unknown",
    "age": 0,
    "role": "Unspecified",
    "salary": 0
})
print("After fillna() with constants:")
df_filled.show()

# Strategy 5: Fill with calculated values (mean/median)
# Calculate mean salary
mean_salary = df_messy.agg({"salary": "mean"}).collect()[0][0]
median_age = df_messy.approxQuantile("age", [0.5], 0.01)[0]

print(f"\nMean salary: ${mean_salary:,.2f}")
print(f"Median age: {median_age}")

df_smart_fill = df_messy.fillna({
    "age": median_age,
    "salary": mean_salary,
    "role": "TBD"
})
df_smart_fill.show()

After fillna() with constants:
+-------+---+-----------+------+
|   name|age|       role|salary|
+-------+---+-----------+------+
|  Alice| 34|   Engineer| 75000|
|    Bob|  0|    Manager| 85000|
|Charlie| 28|Unspecified| 65000|
|  Diana| 45|   Director|     0|
|    Eve|  0|Unspecified|     0|
|  Frank| 32|   Engineer| 70000|
|Unknown| 29|    Analyst| 60000|
+-------+---+-----------+------+


Mean salary: $71,000.00
Median age: 32.0
+-------+---+--------+------+
|   name|age|    role|salary|
+-------+---+--------+------+
|  Alice| 34|Engineer| 75000|
|    Bob| 32| Manager| 85000|
|Charlie| 28|     TBD| 65000|
|  Diana| 45|Director| 71000|
|    Eve| 32|     TBD| 71000|
|  Frank| 32|Engineer| 70000|
|   NULL| 29| Analyst| 60000|
+-------+---+--------+------+



### 🎯 Exercise 1: Handle Missing Values

**Task**: Clean a messy employee dataset.

**Requirements**:
1. Drop rows where both `name` AND `email` are null
2. Fill missing `department` with "Unassigned"
3. Fill missing `salary` with the median salary
4. Count how many rows were affected

In [10]:
# Exercise 1: Your messy dataset
exercise_data = [
    ("John Doe", "john@company.com", "Sales", 55000),
    ("Jane Smith", None, "Engineering", 75000),
    (None, None, "Marketing", 60000),  # Should be dropped
    ("Bob Jones", "bob@company.com", None, 65000),
    ("Alice Wong", "alice@company.com", "Engineering", None),
    ("Charlie", "charlie@company.com", None, None),
    (None, "anon@company.com", "Sales", 58000)
]

df_exercise = spark.createDataFrame(exercise_data, ["name", "email", "department", "salary"])
print("Original messy data:")
df_exercise.show()

# YOUR CODE HERE:
# Step 1: Drop rows where BOTH name AND email are null
df_clean = df_exercise.dropna(subset=["name", "email"], how="all")

# Step 2: Fill missing department
df_clean = df_clean.fillna({"department": "Unassigned"})

# Step 3: Fill missing salary with median
median_salary = df_exercise.approxQuantile("salary", [0.5], 0.01)[0]
df_clean = df_clean.fillna({"salary": median_salary})

print(f"\nCleaned data ({df_clean.count()} rows from {df_exercise.count()}):")
df_clean.show()

Original messy data:
+----------+-------------------+-----------+------+
|      name|              email| department|salary|
+----------+-------------------+-----------+------+
|  John Doe|   john@company.com|      Sales| 55000|
|Jane Smith|               NULL|Engineering| 75000|
|      NULL|               NULL|  Marketing| 60000|
| Bob Jones|    bob@company.com|       NULL| 65000|
|Alice Wong|  alice@company.com|Engineering|  NULL|
|   Charlie|charlie@company.com|       NULL|  NULL|
|      NULL|   anon@company.com|      Sales| 58000|
+----------+-------------------+-----------+------+


Cleaned data (6 rows from 7):
+----------+-------------------+-----------+------+
|      name|              email| department|salary|
+----------+-------------------+-----------+------+
|  John Doe|   john@company.com|      Sales| 55000|
|Jane Smith|               NULL|Engineering| 75000|
| Bob Jones|    bob@company.com| Unassigned| 65000|
|Alice Wong|  alice@company.com|Engineering| 60000|
|   Charlie

<a id="4"></a>
## 4️⃣ Duplicate Detection and Removal

### Finding Duplicates

Duplicates can be:
- **Exact**: All columns match
- **Partial**: Key columns match (e.g., same email, different name)
- **Fuzzy**: Similar but not identical (future topic)

### Methods:

```python
# Remove ALL duplicate rows
df.dropDuplicates()

# Remove duplicates based on specific columns
df.dropDuplicates(["email", "phone"])

# Find duplicate rows for investigation
df.groupBy("email").count().filter("count > 1")
```

In [11]:
# Create dataset with duplicates
dup_data = [
    (1, "alice@email.com", "Alice", "Sales"),
    (2, "bob@email.com", "Bob", "Engineering"),
    (3, "alice@email.com", "Alice", "Sales"),      # Exact duplicate
    (4, "charlie@email.com", "Charlie", "Marketing"),
    (5, "bob@email.com", "Robert", "Engineering"), # Partial duplicate (same email)
    (6, "diana@email.com", "Diana", "Sales"),
    (7, "alice@email.com", "Alice", "Sales")       # Another exact duplicate
]

df_dups = spark.createDataFrame(dup_data, ["id", "email", "name", "department"])
print(f"Original data: {df_dups.count()} rows")
df_dups.show()

Original data: 7 rows
+---+-----------------+-------+-----------+
| id|            email|   name| department|
+---+-----------------+-------+-----------+
|  1|  alice@email.com|  Alice|      Sales|
|  2|    bob@email.com|    Bob|Engineering|
|  3|  alice@email.com|  Alice|      Sales|
|  4|charlie@email.com|Charlie|  Marketing|
|  5|    bob@email.com| Robert|Engineering|
|  6|  diana@email.com|  Diana|      Sales|
|  7|  alice@email.com|  Alice|      Sales|
+---+-----------------+-------+-----------+



In [12]:
# Method 1: Find duplicates for investigation
print("\nFinding duplicate emails:")
duplicates = df_dups.groupBy("email") \
    .count() \
    .filter("count > 1") \
    .orderBy(col("count").desc())

duplicates.show()

# Show all rows for duplicate emails
print("\nAll rows with duplicate emails:")
dup_emails = duplicates.select("email").rdd.flatMap(lambda x: x).collect()
df_dups.filter(col("email").isin(dup_emails)).orderBy("email").show()


Finding duplicate emails:
+---------------+-----+
|          email|count|
+---------------+-----+
|alice@email.com|    3|
|  bob@email.com|    2|
+---------------+-----+


All rows with duplicate emails:
+---+---------------+------+-----------+
| id|          email|  name| department|
+---+---------------+------+-----------+
|  7|alice@email.com| Alice|      Sales|
|  1|alice@email.com| Alice|      Sales|
|  3|alice@email.com| Alice|      Sales|
|  5|  bob@email.com|Robert|Engineering|
|  2|  bob@email.com|   Bob|Engineering|
+---+---------------+------+-----------+



In [13]:
# Method 2: Remove ALL exact duplicates
df_no_exact_dups = df_dups.dropDuplicates()
print(f"\nAfter removing exact duplicates: {df_no_exact_dups.count()} rows")
df_no_exact_dups.show()

# Method 3: Remove duplicates based on email only (keeps first occurrence)
df_unique_emails = df_dups.dropDuplicates(["email"])
print(f"\nAfter removing duplicate emails: {df_unique_emails.count()} rows")
df_unique_emails.orderBy("email").show()

# Method 4: Remove duplicates based on multiple columns
df_unique_combo = df_dups.dropDuplicates(["email", "department"])
print(f"\nUnique email+department combinations: {df_unique_combo.count()} rows")
df_unique_combo.orderBy("email").show()


After removing exact duplicates: 7 rows
+---+-----------------+-------+-----------+
| id|            email|   name| department|
+---+-----------------+-------+-----------+
|  3|  alice@email.com|  Alice|      Sales|
|  2|    bob@email.com|    Bob|Engineering|
|  1|  alice@email.com|  Alice|      Sales|
|  6|  diana@email.com|  Diana|      Sales|
|  7|  alice@email.com|  Alice|      Sales|
|  5|    bob@email.com| Robert|Engineering|
|  4|charlie@email.com|Charlie|  Marketing|
+---+-----------------+-------+-----------+


After removing duplicate emails: 4 rows
+---+-----------------+-------+-----------+
| id|            email|   name| department|
+---+-----------------+-------+-----------+
|  1|  alice@email.com|  Alice|      Sales|
|  2|    bob@email.com|    Bob|Engineering|
|  4|charlie@email.com|Charlie|  Marketing|
|  6|  diana@email.com|  Diana|      Sales|
+---+-----------------+-------+-----------+


Unique email+department combinations: 4 rows
+---+-----------------+-------+---

### 📝 Exercise 2: Remove Duplicates with Window Functions

**Task**: Keep most recent customer record per email.

**Requirements**:
1. Find duplicate emails
2. Use Window function to rank by date
3. Keep only the most recent record per email

In [14]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

customer_data = [
    (1, "alice@email.com", "Alice Johnson", "2023-01-15"),
    (2, "bob@email.com", "Bob Smith", "2023-02-20"),
    (3, "alice@email.com", "Alice J.", "2023-03-10"),  # Newer
    (4, "charlie@email.com", "Charlie", "2023-01-05"),
]

df_customers = spark.createDataFrame(customer_data, ["id", "email", "name", "date"])

print(f"Original: {df_customers.count()} customers")
df_customers.show()

# YOUR CODE HERE:
# Step 1: Create window spec partitioned by email, ordered by date desc
windowSpec = Window.partitionBy('email').orderBy(col('date').desc())

# Step 2: Add rank column
df_ranked = df_customers.withColumn('rank', row_number().over(windowSpec))

# Step 3: Keep only rank = 1
df_unique = df_ranked.filter('rank=1')

print(f"\nCleaned: {df_unique.count()} customers")
df_unique.show()

Original: 4 customers
+---+-----------------+-------------+----------+
| id|            email|         name|      date|
+---+-----------------+-------------+----------+
|  1|  alice@email.com|Alice Johnson|2023-01-15|
|  2|    bob@email.com|    Bob Smith|2023-02-20|
|  3|  alice@email.com|     Alice J.|2023-03-10|
|  4|charlie@email.com|      Charlie|2023-01-05|
+---+-----------------+-------------+----------+


Cleaned: 3 customers
+---+-----------------+---------+----------+----+
| id|            email|     name|      date|rank|
+---+-----------------+---------+----------+----+
|  3|  alice@email.com| Alice J.|2023-03-10|   1|
|  2|    bob@email.com|Bob Smith|2023-02-20|   1|
|  4|charlie@email.com|  Charlie|2023-01-05|   1|
+---+-----------------+---------+----------+----+



In [ ]:
# ✅ Solution
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

windowSpec = Window.partitionBy("email").orderBy(col("date").desc())

df_ranked = df_customers.withColumn("rank", row_number().over(windowSpec))

df_unique = df_ranked.filter(col("rank") == 1).drop("rank")

print(f"Cleaned: {df_unique.count()} customers")
df_unique.show()

<a id="5"></a>
## 5️⃣ Data Type Conversions and Validation

### Common Type Conversions:

| **From** | **To** | **Method** |
|----------|--------|------------|
| String | Integer | `cast(IntegerType())` |
| String | Date | `to_date()` |
| String | Timestamp | `to_timestamp()` |
| Integer | String | `cast(StringType())` |
| Any | Boolean | `cast(BooleanType())` |

### Handling Conversion Errors:

When casting fails, Spark returns `null` - always validate!

In [15]:
# Create data with type issues
type_data = [
    ("1", "100", "2023-01-15", "true"),
    ("2", "200", "2023-02-20", "false"),
    ("3", "abc", "2023-03-25", "1"),      # Bad integer
    ("4", "400", "not-a-date", "true"),   # Bad date
    ("5", "500.50", "2023-05-30", "yes")  # Decimal as string
]

df_types = spark.createDataFrame(type_data, ["id_str", "amount_str", "date_str", "flag_str"])
print("Original data (all strings):")
df_types.printSchema()
df_types.show()

Original data (all strings):
root
 |-- id_str: string (nullable = true)
 |-- amount_str: string (nullable = true)
 |-- date_str: string (nullable = true)
 |-- flag_str: string (nullable = true)

+------+----------+----------+--------+
|id_str|amount_str|  date_str|flag_str|
+------+----------+----------+--------+
|     1|       100|2023-01-15|    true|
|     2|       200|2023-02-20|   false|
|     3|       abc|2023-03-25|       1|
|     4|       400|not-a-date|    true|
|     5|    500.50|2023-05-30|     yes|
+------+----------+----------+--------+



In [24]:
from pyspark.sql.functions import try_to_timestamp, lit

# Safe type conversions with validation
df_converted = df_types \
    .withColumn("id", col("id_str").cast(IntegerType())) \
    .withColumn("amount", col("amount_str").try_cast(DoubleType())) \
    .withColumn("date", try_to_timestamp(col("date_str"), lit("yyyy-MM-dd")).cast(DateType())) \
    .withColumn("flag",
                when(col("flag_str").isin("true", "1", "yes"), True)
                .when(col("flag_str").isin("false", "0", "no"), False)
                .otherwise(None))

print("\nAfter type conversions:")
df_converted.printSchema()
df_converted.show()

# Check for conversion failures (nulls)
print("\nChecking for conversion failures:")
df_converted.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in ["id", "amount", "date", "flag"]
]).show()


After type conversions:
root
 |-- id_str: string (nullable = true)
 |-- amount_str: string (nullable = true)
 |-- date_str: string (nullable = true)
 |-- flag_str: string (nullable = true)
 |-- id: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- date: date (nullable = true)
 |-- flag: boolean (nullable = true)

+------+----------+----------+--------+---+------+----------+-----+
|id_str|amount_str|  date_str|flag_str| id|amount|      date| flag|
+------+----------+----------+--------+---+------+----------+-----+
|     1|       100|2023-01-15|    true|  1| 100.0|2023-01-15| true|
|     2|       200|2023-02-20|   false|  2| 200.0|2023-02-20|false|
|     3|       abc|2023-03-25|       1|  3|  NULL|2023-03-25| true|
|     4|       400|not-a-date|    true|  4| 400.0|      NULL| true|
|     5|    500.50|2023-05-30|     yes|  5| 500.5|2023-05-30| true|
+------+----------+----------+--------+---+------+----------+-----+


Checking for conversion failures:
+---+------+----+

<a id="6"></a>
## 6️⃣ String Cleaning and Manipulation

### Common String Operations:

| **Operation** | **Function** | **Example** |
|---------------|--------------|-------------|
| Trim whitespace | `trim()`, `ltrim()`, `rtrim()` | `" hello " → "hello"` |
| Change case | `upper()`, `lower()`, `initcap()` | `"HeLLo" → "hello"` |
| Replace text | `regexp_replace()` | Remove special chars |
| Extract pattern | `regexp_extract()` | Extract phone digits |
| Split string | `split()` | `"a,b,c" → ["a","b","c"]` |
| Concatenate | `concat()`, `concat_ws()` | Combine columns |

In [25]:
# Create messy string data
string_data = [
    ("  ALICE JOHNSON  ", "alice@EMAIL.com ", "(555) 123-4567"),
    ("bob smith", " BOB@email.COM", "555-234-5678  "),
    ("  Charlie_Brown  ", "charlie@email.com", "5552345678"),
    ("DIANA prince", "diana@EMAIL.com  ", "(555)-345-6789")
]

df_strings = spark.createDataFrame(string_data, ["name", "email", "phone"])
print("Original messy strings:")
df_strings.show(truncate=False)

Original messy strings:
+-----------------+-----------------+--------------+
|name             |email            |phone         |
+-----------------+-----------------+--------------+
|  ALICE JOHNSON  |alice@EMAIL.com  |(555) 123-4567|
|bob smith        | BOB@email.COM   |555-234-5678  |
|  Charlie_Brown  |charlie@email.com|5552345678    |
|DIANA prince     |diana@EMAIL.com  |(555)-345-6789|
+-----------------+-----------------+--------------+



In [26]:
# Clean strings step by step
df_clean_strings = df_strings \
    .withColumn("name_clean",
                initcap(trim(regexp_replace(col("name"), "_", " ")))) \
    .withColumn("email_clean",
                lower(trim(col("email")))) \
    .withColumn("phone_clean",
                regexp_replace(col("phone"), "[^0-9]", ""))  # Keep only digits

print("\nCleaned strings:")
df_clean_strings.select("name_clean", "email_clean", "phone_clean").show(truncate=False)

# Advanced: Split name into first and last
df_clean_strings = df_clean_strings \
    .withColumn("name_parts", split(col("name_clean"), " ")) \
    .withColumn("first_name", col("name_parts").getItem(0)) \
    .withColumn("last_name", col("name_parts").getItem(size(col("name_parts")) - 1)) \
    .drop("name_parts")

df_clean_strings.select("name_clean", "first_name", "last_name").show(truncate=False)


Cleaned strings:
+-------------+-----------------+-----------+
|name_clean   |email_clean      |phone_clean|
+-------------+-----------------+-----------+
|Alice Johnson|alice@email.com  |5551234567 |
|Bob Smith    |bob@email.com    |5552345678 |
|Charlie Brown|charlie@email.com|5552345678 |
|Diana Prince |diana@email.com  |5553456789 |
+-------------+-----------------+-----------+



/usr/local/lib/python3.13/dist-packages/pyspark/sql/classic/column.py:359: FutureWarning: A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.
  warnings.warn(


+-------------+----------+---------+
|name_clean   |first_name|last_name|
+-------------+----------+---------+
|Alice Johnson|Alice     |Johnson  |
|Bob Smith    |Bob       |Smith    |
|Charlie Brown|Charlie   |Brown    |
|Diana Prince |Diana     |Prince   |
+-------------+----------+---------+



### 📝 Exercise 3: Clean Phone Numbers

**Task**: Standardize phone numbers to 10 digits.

**Requirements**:
1. Extract only digits from phone numbers
2. Validate length is exactly 10 digits
3. Flag invalid phone numbers

In [34]:
contact_data = [
    ("Alice", "(555) 123-4567"),
    ("Bob", "555.234.5678"),
    ("Charlie", "555-345-67"),  # Too short
]

df_contacts = spark.createDataFrame(contact_data, ["name", "phone"])

print("Original:")
df_contacts.show()

# YOUR CODE HERE:
# Step 1: Extract digits only
df_clean = df_contacts.withColumn("phone_clean", regexp_replace(col('phone'), '[^0-9]', ''))

# Step 2: Validate length = 10
df_clean = df_clean.withColumn("is_valid", size(split(col('phone_clean'), ''))==10)

print("\nCleaned:")
df_clean.show()

Original:
+-------+--------------+
|   name|         phone|
+-------+--------------+
|  Alice|(555) 123-4567|
|    Bob|  555.234.5678|
|Charlie|    555-345-67|
+-------+--------------+


Cleaned:
+-------+--------------+-----------+--------+
|   name|         phone|phone_clean|is_valid|
+-------+--------------+-----------+--------+
|  Alice|(555) 123-4567| 5551234567|    true|
|    Bob|  555.234.5678| 5552345678|    true|
|Charlie|    555-345-67|   55534567|   false|
+-------+--------------+-----------+--------+



In [35]:
# ✅ Solution
df_clean = df_contacts.withColumn("phone_clean",
    regexp_replace(col("phone"), "[^0-9]", ""))

df_clean = df_clean.withColumn("is_valid",
    when(length(col("phone_clean")) == 10, True).otherwise(False))

print("Cleaned:")
df_clean.show()

print("\nInvalid phones:")
df_clean.filter(~col("is_valid")).show()

Cleaned:
+-------+--------------+-----------+--------+
|   name|         phone|phone_clean|is_valid|
+-------+--------------+-----------+--------+
|  Alice|(555) 123-4567| 5551234567|    true|
|    Bob|  555.234.5678| 5552345678|    true|
|Charlie|    555-345-67|   55534567|   false|
+-------+--------------+-----------+--------+


Invalid phones:
+-------+----------+-----------+--------+
|   name|     phone|phone_clean|is_valid|
+-------+----------+-----------+--------+
|Charlie|555-345-67|   55534567|   false|
+-------+----------+-----------+--------+



### 🎯 Exercise 2: Clean Voter Data

**Task**: Clean a voter registration dataset.

**Requirements**:
1. Filter out names that are < 2 or > 50 characters
2. Filter out names containing underscore or numbers
3. Split VOTER_NAME into first_name and last_name
4. Normalize names to proper case (initcap)

In [3]:
# Exercise 2: Voter data
voter_data = [
    ("ALICE MARIE JOHNSON",),
    ("bob_smith",),              # Has underscore - remove
    ("CHARLIE BROWN III",),
    ("X",),                      # Too short - remove
    ("Diana Prince",),
    ("_INVALID_NAME_",),         # Has underscore - remove
    ("Frank James Miller Jr",),
    ("123_Test_User",)           # Has numbers - remove
]

df_voters_ex = spark.createDataFrame(voter_data, ["VOTER_NAME"])
print(f"Original voter data: {df_voters_ex.count()} rows")
df_voters_ex.show(truncate=False)

# YOUR CODE HERE:
# Step 1: Filter by length
df_voters_clean = df_voters_ex.filter(
    (length(col("VOTER_NAME")) >= 2) & (length(col("VOTER_NAME")) <= 50)
)

# Step 2: Filter out underscore and numbers
df_voters_clean = df_voters_clean.filter(
    ~col("VOTER_NAME").contains("_")
).filter(
    ~col("VOTER_NAME").rlike("[0-9]")
)

# Step 3 & 4: Split and normalize
df_voters_clean = df_voters_clean \
    .withColumn("splits", split(col("VOTER_NAME"), "\\s+")) \
    .withColumn("first_name", initcap(col("splits").getItem(0))) \
    .withColumn("last_name", initcap(col("splits").getItem(size(col("splits")) - 1))) \
    .drop("splits")

print(f"\nCleaned voter data: {df_voters_clean.count()} rows")
df_voters_clean.show(truncate=False)

Original voter data: 8 rows
+---------------------+
|VOTER_NAME           |
+---------------------+
|ALICE MARIE JOHNSON  |
|bob_smith            |
|CHARLIE BROWN III    |
|X                    |
|Diana Prince         |
|_INVALID_NAME_       |
|Frank James Miller Jr|
|123_Test_User        |
+---------------------+



/usr/local/lib/python3.13/dist-packages/pyspark/sql/classic/column.py:359: FutureWarning: A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.
  warnings.warn(



Cleaned voter data: 4 rows
+---------------------+----------+---------+
|VOTER_NAME           |first_name|last_name|
+---------------------+----------+---------+
|ALICE MARIE JOHNSON  |Alice     |Johnson  |
|CHARLIE BROWN III    |Charlie   |Iii      |
|Diana Prince         |Diana     |Prince   |
|Frank James Miller Jr|Frank     |Jr       |
+---------------------+----------+---------+



<a id="7"></a>
## 7️⃣ User-Defined Functions (UDFs)

### When to Use UDFs:

✅ **Use UDFs when**:
- Built-in functions can't handle your logic
- Complex business rules
- Custom calculations

❌ **Avoid UDFs when**:
- Built-in functions exist (they're faster!)
- Simple operations

### Performance:
- Built-in functions: **10-100x faster**
- UDFs have serialization overhead
- Use Pandas UDFs for better performance (vectorized)

In [4]:
# Example: Reverse string UDF
def reverse_string(s):
    """Reverse a string"""
    if s is None:
        return None
    return s[::-1]

# Register as UDF
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

udf_reverse = udf(reverse_string, StringType())

# Test it
test_data = [("hello",), ("world",), ("spark",), (None,)]
df_test = spark.createDataFrame(test_data, ["word"])

df_test = df_test.withColumn("reversed", udf_reverse(col("word")))
df_test.show()

+-----+--------+
| word|reversed|
+-----+--------+
|hello|   olleh|
|world|   dlrow|
|spark|   kraps|
| NULL|    NULL|
+-----+--------+



In [5]:
# Example: Complex validation UDF
def validate_email(email):
    """
    Simple email validation
    Returns True if valid, False otherwise
    """
    if email is None:
        return False

    import re
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    return bool(re.match(pattern, email))

# Register UDF
from pyspark.sql.types import BooleanType
udf_validate_email = udf(validate_email, BooleanType())

# Test it
email_data = [
    ("alice@company.com",),
    ("invalid.email",),
    ("bob@test.co.uk",),
    ("not_an_email",),
    (None,)
]

df_emails = spark.createDataFrame(email_data, ["email"])
df_emails = df_emails.withColumn("is_valid", udf_validate_email(col("email")))
df_emails.show()

print(f"\nValid emails: {df_emails.filter(col('is_valid')).count()}")

+-----------------+--------+
|            email|is_valid|
+-----------------+--------+
|alice@company.com|    true|
|    invalid.email|   false|
|   bob@test.co.uk|    true|
|     not_an_email|   false|
|             NULL|   false|
+-----------------+--------+


Valid emails: 2


### 📝 Exercise 4: Email Validation UDF

**Task**: Create a UDF to validate email addresses.

**Requirements**:
1. Create a validation function using regex
2. Register as UDF
3. Apply to dataset and count valid emails

In [8]:
email_data = [
    ("alice@company.com",),
    ("invalid.email",),
    ("bob@test.co.uk",),
    ("not_an_email",),
]

df_emails = spark.createDataFrame(email_data, ["email"])

# YOUR CODE HERE:
# Step 1: Create validation function
def validate_email(email):
    if email is None:
        return False
    import re
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    return re.match(pattern, email) is not None

# Step 2: Register UDF
from pyspark.sql.types import BooleanType
validate_email_udf = udf(validate_email, BooleanType())

# Step 3: Apply
df_validated = df_emails.withColumn('is_valid', validate_email_udf('email'))

print("Results:")
df_validated.show()
print(f"\nValid: {df_validated.filter(col('is_valid')).count()}")

Results:
+-----------------+--------+
|            email|is_valid|
+-----------------+--------+
|alice@company.com|    true|
|    invalid.email|   false|
|   bob@test.co.uk|    true|
|     not_an_email|   false|
+-----------------+--------+


Valid: 2


In [9]:
# ✅ Solution
from pyspark.sql.functions import udf
from pyspark.sql.types import BooleanType
import re

def validate_email(email):
    if email is None:
        return False
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    return bool(re.match(pattern, email))

validate_email_udf = udf(validate_email, BooleanType())

df_validated = df_emails.withColumn("is_valid", validate_email_udf(col("email")))

print("Results:")
df_validated.show()
print(f"\nValid: {df_validated.filter(col('is_valid')).count()}")

Results:
+-----------------+--------+
|            email|is_valid|
+-----------------+--------+
|alice@company.com|    true|
|    invalid.email|   false|
|   bob@test.co.uk|    true|
|     not_an_email|   false|
+-----------------+--------+


Valid: 2


<a id="8"></a>
## 8️⃣ Performance Optimization for Cleaning

### Caching Strategy:

```python
# Cache if you'll use DataFrame multiple times
df.cache()

# Check if cached
df.is_cached

# Remove from cache when done
df.unpersist()
```

### Performance Tips:

| **Tip** | **Why** | **Example** |
|---------|---------|-------------|
| **Filter early** | Reduce data volume | Filter before join |
| **Use built-ins** | Avoid UDF overhead | `lower()` vs UDF |
| **Cache smartly** | Reuse expensive computations | After complex transformations |
| **Broadcast small tables** | Avoid shuffle | `broadcast(small_df)` |
| **Repartition wisely** | Balance parallelism | `repartition(200)` |

In [20]:
# Performance comparison: Built-in vs UDF
import time

# Create larger dataset
large_data = [(f"User{i}", f"user{i}@email.com") for i in range(100000)]
df_perf = spark.createDataFrame(large_data, ["name", "email"])

# Method 1: Built-in function
start = time.time()
df_builtin = df_perf.withColumn("email_upper", upper(col("email")))
count1 = df_builtin.count()
builtin_time = time.time() - start

# Method 2: UDF
udf_upper = udf(lambda s: s.upper() if s else None, StringType())
start = time.time()
df_udf = df_perf.withColumn("email_upper", udf_upper(col("email")))
count2 = df_udf.count()
udf_time = time.time() - start

print(f"Built-in function: {builtin_time:.3f}s")
print(f"UDF function: {udf_time:.3f}s")
print(f"Built-in is {udf_time/builtin_time:.1f}x faster!")

Built-in function: 0.401s
UDF function: 0.341s
Built-in is 0.8x faster!


In [25]:
# Caching demonstration
# Create DataFrame with expensive operations
df_expensive = spark.range(0, 100000) \
    .withColumn("squared", col("id") * col("id")) \
    .withColumn("cubed", col("id") * col("id") * col("id"))

# Without caching - runs full computation each time
start = time.time()
count1 = df_expensive.count()
count2 = df_expensive.count()
no_cache_time = time.time() - start

# With caching
df_expensive.cache()
start = time.time()
count1 = df_expensive.count()  # Caches here
count2 = df_expensive.count()  # Uses cache
cache_time = time.time() - start

print(f"\nWithout cache: {no_cache_time:.3f}s")
print(f"With cache: {cache_time:.3f}s")
print(f"Speedup: {no_cache_time/cache_time:.1f}x")

# Clean up
df_expensive.unpersist()


Without cache: 0.135s
With cache: 0.728s
Speedup: 0.2x


DataFrame[id: bigint, squared: bigint, cubed: bigint]

<a id="9"></a>
## 9️⃣ Complete Data Cleaning Pipeline

### Pipeline Steps:

```
1. LOAD with Schema    → Validate structure
2. INSPECT            → Profile data quality  
3. CLEAN             → Fix issues
4. VALIDATE          → Verify fixes
5. SAVE              → Export clean data
```

### Best Practices:

✅ Define schema upfront
✅ Log data quality metrics
✅ Handle errors gracefully  
✅ Cache strategically
✅ Document transformations
✅ Version your data

In [26]:
# Complete cleaning pipeline example
print("="*60)
print("COMPLETE DATA CLEANING PIPELINE")
print("="*60)

# STEP 1: Define schema
employee_schema = StructType([
    StructField("id", StringType(), False),
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("department", StringType(), True),
    StructField("salary", StringType(), True),  # String because it may have errors
    StructField("hire_date", StringType(), True)
])

# STEP 2: Create messy dataset
messy_employee_data = [
    ("1", "  ALICE JOHNSON  ", "alice@company.com", "Sales", "75000", "2020-01-15"),
    ("2", "bob_smith", "invalid.email", "Engineering", "abc", "2019-03-20"),
    ("3", None, "charlie@company.com", None, "65000", "not-a-date"),
    ("1", "  Alice Johnson  ", "alice@company.com", "Sales", "75000", "2020-01-15"),  # Duplicate
    ("4", "Diana Prince", "diana@company.com", "Marketing", "70000", "2021-06-10"),
    ("5", "X", "eve@company.com", "Engineering", "-50000", "2022-02-28"),  # Invalid
    ("6", "Frank Miller", "frank@company.com", None, "68000", "2020-08-15")
]

df_messy_employees = spark.createDataFrame(messy_employee_data, schema=employee_schema)

print(f"\nSTEP 1: Loaded {df_messy_employees.count()} rows")
df_messy_employees.show(truncate=False)

COMPLETE DATA CLEANING PIPELINE

STEP 1: Loaded 7 rows
+---+-----------------+-------------------+-----------+------+----------+
|id |name             |email              |department |salary|hire_date |
+---+-----------------+-------------------+-----------+------+----------+
|1  |  ALICE JOHNSON  |alice@company.com  |Sales      |75000 |2020-01-15|
|2  |bob_smith        |invalid.email      |Engineering|abc   |2019-03-20|
|3  |NULL             |charlie@company.com|NULL       |65000 |not-a-date|
|1  |  Alice Johnson  |alice@company.com  |Sales      |75000 |2020-01-15|
|4  |Diana Prince     |diana@company.com  |Marketing  |70000 |2021-06-10|
|5  |X                |eve@company.com    |Engineering|-50000|2022-02-28|
|6  |Frank Miller     |frank@company.com  |NULL       |68000 |2020-08-15|
+---+-----------------+-------------------+-----------+------+----------+



In [27]:
# STEP 3: Inspect data quality
print("\n" + "="*60)
print("STEP 2: DATA QUALITY INSPECTION")
print("="*60)

# Check nulls
print("\nNull counts:")
df_messy_employees.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_messy_employees.columns
]).show()

# Check duplicates
print(f"\nDuplicate IDs: {df_messy_employees.groupBy('id').count().filter('count > 1').count()}")

# Check data types
print("\nSchema:")
df_messy_employees.printSchema()


STEP 2: DATA QUALITY INSPECTION

Null counts:
+---+----+-----+----------+------+---------+
| id|name|email|department|salary|hire_date|
+---+----+-----+----------+------+---------+
|  0|   1|    0|         2|     0|        0|
+---+----+-----+----------+------+---------+


Duplicate IDs: 1

Schema:
root
 |-- id: string (nullable = false)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: string (nullable = true)
 |-- hire_date: string (nullable = true)



In [29]:
# STEP 4: Clean the data
print("\n" + "="*60)
print("STEP 3: DATA CLEANING")
print("="*60)

# Remove duplicates
df_clean = df_messy_employees.dropDuplicates(["id"])
print(f"After deduplication: {df_clean.count()} rows")

# Clean strings
df_clean = df_clean \
    .withColumn("name", trim(initcap(regexp_replace(col("name"), "_", " ")))) \
    .withColumn("email", lower(trim(col("email")))) \
    .withColumn("department", trim(col("department")))

# Validate and filter names (length between 5-50)
df_clean = df_clean.filter(
    (length(col("name")) >= 5) & (length(col("name")) <= 50)
)
print(f"After name validation: {df_clean.count()} rows")

# Type conversions with validation
df_clean = df_clean \
    .withColumn("salary_int", expr("try_cast(salary as int)")) \
    .withColumn("hire_date_parsed", to_date(col("hire_date"), "yyyy-MM-dd"))

# Filter invalid salaries (must be positive and converted successfully)
df_clean = df_clean.filter(
    (col("salary_int").isNotNull()) & (col("salary_int") > 0)
)
print(f"After salary validation: {df_clean.count()} rows")

# Handle missing departments
df_clean = df_clean.fillna({"department": "Unassigned"})

# Drop original columns, keep cleaned versions
df_clean = df_clean \
    .withColumn("id", col("id").cast(IntegerType())) \
    .drop("salary", "hire_date") \
    .withColumnRenamed("salary_int", "salary") \
    .withColumnRenamed("hire_date_parsed", "hire_date")

print("\nFinal cleaned data:")
df_clean.show(truncate=False)
df_clean.printSchema()


STEP 3: DATA CLEANING
After deduplication: 6 rows
After name validation: 4 rows
After salary validation: 3 rows

Final cleaned data:
+---+-------------+-----------------+----------+------+----------+
|id |name         |email            |department|salary|hire_date |
+---+-------------+-----------------+----------+------+----------+
|1  |Alice Johnson|alice@company.com|Sales     |75000 |2020-01-15|
|4  |Diana Prince |diana@company.com|Marketing |70000 |2021-06-10|
|6  |Frank Miller |frank@company.com|Unassigned|68000 |2020-08-15|
+---+-------------+-----------------+----------+------+----------+

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- department: string (nullable = false)
 |-- salary: integer (nullable = true)
 |-- hire_date: date (nullable = true)



In [30]:
# STEP 5: Validate cleaned data
print("\n" + "="*60)
print("STEP 4: VALIDATION")
print("="*60)

# Validation checks
print("✓ No duplicate IDs:", df_clean.select("id").distinct().count() == df_clean.count())
print("✓ No null names:", df_clean.filter(col("name").isNull()).count() == 0)
print("✓ All valid salaries:", df_clean.filter(col("salary") <= 0).count() == 0)
print("✓ No null departments:", df_clean.filter(col("department").isNull()).count() == 0)

# Summary stats
print("\nSummary Statistics:")
df_clean.select("salary").summary().show()

# STEP 6: Save (would write to parquet in production)
# df_clean.write.parquet("cleaned_employees.parquet", mode="overwrite")
print("\n✅ Pipeline complete! Data is ready for analysis.")


STEP 4: VALIDATION
✓ No duplicate IDs: True
✓ No null names: True
✓ All valid salaries: True
✓ No null departments: True

Summary Statistics:
+-------+------------------+
|summary|            salary|
+-------+------------------+
|  count|                 3|
|   mean|           71000.0|
| stddev|3605.5512754639894|
|    min|             68000|
|    25%|             68000|
|    50%|             70000|
|    75%|             75000|
|    max|             75000|
+-------+------------------+


✅ Pipeline complete! Data is ready for analysis.


### 📝 Exercise 5: Complete Cleaning Pipeline

**Task**: Build end-to-end pipeline for sales data.

**Requirements**:
1. Remove duplicates by transaction_id
2. Drop rows with null amount
3. Convert date strings and validate
4. Filter out negative amounts
5. Calculate summary statistics

In [ ]:
sales_data = [
    ("TXN001", "2023-01-15", "Alice", "1200.50"),
    ("TXN001", "2023-01-15", "Alice", "1200.50"),  # Duplicate
    ("TXN002", "2023-01-16", "Bob", "450.00"),
    ("TXN003", "bad-date", "Charlie", "75.25"),
    ("TXN004", "2023-01-17", "Diana", None),  # Null
    ("TXN005", "2023-01-18", "Eve", "-50.00"),  # Negative
]

df_sales = spark.createDataFrame(sales_data, ["txn_id", "date_str", "customer", "amount_str"])

print(f"Original: {df_sales.count()} rows")

# YOUR CODE HERE:
# Step 1: Remove duplicates
df_clean = df_s

# Step 2: Drop nulls
df_clean = ____

# Step 3: Convert types
df_clean = df_clean \
    .withColumn("date", to_date(col("date_str"), "yyyy-MM-dd")) \
    .withColumn("amount", col("amount_str").cast(DoubleType()))

# Step 4: Filter valid (date not null, amount > 0)
df_clean = ____

# Step 5: Summary
print(f"\nCleaned: {df_clean.count()} rows")
df_clean.select("txn_id", "date", "customer", "amount").show()
df_clean.agg({"amount": "sum"}).show()

In [ ]:
# ✅ Solution
df_clean = df_sales.dropDuplicates(["txn_id"])

df_clean = df_clean.dropna(subset=["amount_str"])

df_clean = df_clean \
    .withColumn("date", to_date(col("date_str"), "yyyy-MM-dd")) \
    .withColumn("amount", col("amount_str").cast(DoubleType()))

df_clean = df_clean \
    .filter(col("date").isNotNull()) \
    .filter(col("amount") > 0)

print(f"Cleaned: {df_clean.count()} rows")
df_clean.select("txn_id", "date", "customer", "amount").show()

print("\nTotal sales:")
df_clean.agg({"amount": "sum"}).show()

## 🎓 Summary and Best Practices

### What You've Learned:

✅ **Data Quality Fundamentals**:
- Why cleaning matters (cost of dirty data)
- Common data quality issues

✅ **Core Cleaning Techniques**:
- Schema definition and enforcement
- Missing data strategies (drop, fill, impute)
- Duplicate detection and removal
- Type conversions and validation
- String cleaning (trim, case, regex)

✅ **Advanced Techniques**:
- User-Defined Functions (UDFs)
- Performance optimization
- Caching strategies

✅ **Production Pipeline**:
- End-to-end cleaning workflow
- Data quality validation
- Best practices

### Cleaning Checklist:

```
☐ Define schema upfront
☐ Inspect data quality (nulls, duplicates, types)
☐ Document all transformations
☐ Handle nulls appropriately (drop vs fill)
☐ Remove or handle duplicates
☐ Validate and convert data types
☐ Clean string data (trim, normalize)
☐ Filter invalid/outlier data
☐ Validate cleaned data
☐ Cache expensive operations
☐ Write to optimized format (Parquet)
☐ Log data quality metrics
```

### Key Functions Reference:

| **Task** | **Function** |
|----------|--------------|
| Drop nulls | `dropna()`, `dropna(subset=[...])` |
| Fill nulls | `fillna(...)` |
| Remove duplicates | `dropDuplicates()`, `dropDuplicates([...])` |
| Type conversion | `cast()`, `to_date()`, `to_timestamp()` |
| Trim whitespace | `trim()`, `ltrim()`, `rtrim()` |
| Change case | `upper()`, `lower()`, `initcap()` |
| Regex replace | `regexp_replace()` |
| Regex extract | `regexp_extract()` |
| Split strings | `split()` |
| Custom logic | `udf()` |
| Filter rows | `filter()`, `where()` |
| Conditional logic | `when()`, `otherwise()` |

### Performance Tips:

1. **Use built-in functions** over UDFs (10-100x faster)
2. **Filter early** to reduce data volume
3. **Cache** DataFrames used multiple times
4. **Broadcast** small lookup tables
5. **Partition** appropriately (2-4 per CPU core)
6. **Use Parquet** for intermediate/final storage

### Next Steps:

1. Practice with real messy datasets (Kaggle, UCI)
2. Build reusable cleaning functions library
3. Learn about data quality monitoring
4. Explore ML-based imputation techniques
5. Study advanced regex patterns
6. Learn Pandas UDFs for vectorized operations

### Resources:

- 📖 [PySpark SQL Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql.html)
- 📖 [Data Cleaning Best Practices](https://www.datacamp.com/)
- 💻 [Regex Testing](https://regex101.com/)

---

### 🎉 Congratulations!

You've completed **Data Cleaning with PySpark**! You now have production-ready skills to:
- Clean messy data at scale
- Build robust data quality pipelines
- Optimize performance for large datasets
- Validate and monitor data quality

**Keep practicing with real-world datasets!** 🚀

In [ ]:
# Clean up SparkSession
spark.stop()
print("SparkSession stopped. Notebook complete!")